# Bronze Layer — CSV to Delta Ingestion

This notebook ingests raw CSV files from UC volumes and writes them as Delta tables in the `workspace.bronze` schema.

## Table of Contents

| Step | Source Volume | Table | Rows | Cell |
| --- | --- | --- | --- | --- |
| 1 | `source_crm` | `workspace.bronze.cust_info` | 18,494 | Cell 2 |
| 2 | `source_crm` | `workspace.bronze.prd_info` | 397 | Cell 2 |
| 3 | `source_crm` | `workspace.bronze.sales_details` | 60,398 | Cell 2 |
| 4 | `source_erp` | `workspace.bronze.CUST_AZ12` | 18,484 | Cell 3 |
| 5 | `source_erp` | `workspace.bronze.LOC_A101` | 18,484 | Cell 3 |
| 6 | `source_erp` | `workspace.bronze.PX_CAT_G1V2` | 37 | Cell 3 |

## Process Overview

1. **CRM Ingestion (Cell 2)** — Reads `cust_info`, `prd_info`, and `sales_details` CSVs from `workspace.bronze.source_system/source_crm/` and writes them as Delta tables.
2. **ERP Ingestion (Cell 3)** — Auto-discovers all `.csv` files in `workspace.bronze.source_system/source_erp/` and writes each as a Delta table.

Both steps use `spark.read.csv` with `header=True, inferSchema=True` and write with `mode("overwrite")`.

In [0]:
source_base = "/Volumes/workspace/bronze/source_system/source_crm"
target_schema = "workspace.bronze"

files = {
    "cust_info":      f"{source_base}/cust_info.csv",
    "prd_info":       f"{source_base}/prd_info.csv",
    "sales_details":  f"{source_base}/sales_details.csv",
}

for table_name, csv_path in files.items():
    full_table = f"{target_schema}.{table_name}"
    print(f"Reading {csv_path} -> {full_table} ...")
    df = spark.read.csv(csv_path, header=True, inferSchema=True)
    df.write.format("delta").mode("overwrite").saveAsTable(full_table)
    print(f"  Written {df.count()} rows to {full_table}")

print("\nDone — all CRM CSV files written as Delta tables.")


In [0]:
source_base = "/Volumes/workspace/bronze/source_system/source_erp"
target_schema = "workspace.bronze"

# List the CSV files available in the source_erp volume directory
import os

files = {
    table_name: f"{source_base}/{table_name}.csv"
    for table_name in [
        f.removesuffix(".csv") for f in os.listdir(source_base) if f.endswith(".csv")
    ]
}

for table_name, csv_path in files.items():
    full_table = f"{target_schema}.{table_name}"
    print(f"Reading {csv_path} -> {full_table} ...")
    df = spark.read.csv(csv_path, header=True, inferSchema=True)
    df.write.format("delta").mode("overwrite").saveAsTable(full_table)
    print(f"  Written {df.count()} rows to {full_table}")

print("\nDone — all ERP CSV files written as Delta tables.")